# 🌲 AICOSS 26 Capstone – Tree-Based Document QA Evaluation (v1)

## Architecture

**Key Innovation: LLM-Guided Document Structuring (StructureAgent v2)**

```
PDF → Docling Parse → Raw Tree → StructureAgent (LLM) → Restructured Tree → Summarize → Index → Query
                                       ↑
                          LLM analyzes headings,
                          identifies organizational units,
                          outputs restructuring plan (JSON)
```

**Improvements in v1:**
- **StructureAgent v2**: Majority-voting boundary detection + container node (OHATER) flattening
- **Park-aware ToolBox**: 9 tools including `search_keyword_all`, `get_all_entries`, `extract_from_all`
- **Stronger agent prompt**: NEVER omit results rule, complete tool descriptions
- **JSON parsing hardened**: Strips markdown code blocks before parsing
- **AGENT_MAX_ITER=20**, max_tokens=4096

**Agent Pipeline:**
```
PDF → Docling + Hierarchy Inference → Document Tree → Summaries (Mistral) → ChromaDB
                                                                              ↓
Question → Agent (Mistral Function Calling + 9 Tools) → Tree Navigation → Answer
```

## Evaluation Design
| Type | Description | What it tests |
|------|-------------|---------------|
| **A** Selection | 단일 span/사실 추출 | Precision retrieval |
| **B** Enumeration | 항목 수집·개수 계산 | Full coverage |
| **C** Aggregation | 여러 구간 의미 통합 | Reduce reasoning |
| **D** Multi-hop | 서로 다른 위치 정보 연결 | Hierarchical reasoning |

**3 Documents × 4 Categories × 10 Questions = 120 total evaluations**

## 0. Setup & Installation

## 1. Configuration & Imports

In [ ]:
import os, json, time, pickle, hashlib, shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
import warnings
warnings.filterwarnings("ignore")

# ═══ SET YOUR API KEY HERE ═══

MISTRAL_MODEL   = "mistral-small-latest"
EMBED_MODEL_ID  = "sentence-transformers/all-MiniLM-L6-v2"

JUDGE_PROVIDER = "gemini"  # "gemini" | "groq" | "mistral"


DOCS = {
    "gen_ai": {
        "pdf": "Top_Generative_AI_Use_Cases_You_Should_Know_About_in_2025.pdf",
        "qa":  "gen_ai_usecases_qa.json",
    },
    "ai_act": {
        "pdf": "AI_Act.pdf",
        "qa":  "ai_act_qa.json",
    },
    "parks": {
        "pdf": "national-parks-of-europe.pdf",
        "qa":  "national_parks_qa.json",
    },
}

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
print("Config loaded")

In [ ]:
@dataclass
class DocNode:
    node_id:  str
    title:    str
    text:     str
    level:    int = 0
    kind:     str = "section"
    summary:  str = ""
    children: list = field(default_factory=list)
    meta:     dict = field(default_factory=dict)

    @property
    def is_leaf(self): return len(self.children) == 0

    def all_text(self):
        parts = [self.text] + [c.all_text() for c in self.children]
        return chr(10).join(p for p in parts if p.strip())

    def leaves(self):
        if self.is_leaf: return [self]
        out = []
        for c in self.children: out.extend(c.leaves())
        return out

    def flatten(self):
        out = [self]
        for c in self.children: out.extend(c.flatten())
        return out

    def find(self, nid):
        if self.node_id == nid: return self
        for c in self.children:
            hit = c.find(nid)
            if hit: return hit
        return None

_id_counter = 0
def _id(seed, prefix="n"):
    global _id_counter
    _id_counter += 1
    return f"{prefix}_{hashlib.md5(seed.encode()).hexdigest()[:12]}_{_id_counter}"

print("DocNode defined")

### 2.1 PDF Parser
Uses **Docling** `DocumentConverter` + **docling-hierarchical-pdf** `ResultPostprocessor`.
Fallback: Markdown heading-based tree construction.

In [ ]:
def _md_to_tree(md_text, name):
    root = DocNode(_id(name, "root"), name, "", 0, "root")
    stack = [(0, root)]
    buf = []
    para_n = 0
    NL = chr(10)

    def flush():
        nonlocal buf, para_n
        txt = NL.join(buf).strip()
        if not txt:
            buf.clear()
            return
        parent = stack[-1][1]
        para_n += 1
        leaf = DocNode(_id(f"p{para_n}_{txt[:40]}"), f"Paragraph {para_n}",
                       txt, parent.level + 1, "text")
        parent.children.append(leaf)
        buf.clear()

    for line in md_text.split(NL):
        s = line.strip()
        if s.startswith("#"):
            flush()
            lvl = len(s) - len(s.lstrip("#"))
            heading = s.lstrip("#").strip()
            if not heading:
                continue
            while len(stack) > 1 and stack[-1][0] >= lvl:
                stack.pop()
            sec = DocNode(_id(f"s{lvl}_{heading}"), heading, "", lvl, "section")
            stack[-1][1].children.append(sec)
            stack.append((lvl, sec))
        elif s:
            buf.append(s)
        elif buf:
            flush()
    flush()
    return root


def parse_pdf(pdf_path):
    path = Path(pdf_path)
    print(f"Parsing: {path.name}")

    try:
        from docling.document_converter import DocumentConverter
        converter = DocumentConverter()
        result = converter.convert(str(path))
        print("  Docling conversion done")

        try:
            from hierarchical.postprocessor import ResultPostprocessor
            ResultPostprocessor(result, source=str(path)).process()
            print("  Hierarchy inference applied")
        except Exception as e:
            print(f"  Hierarchy skip: {e}")

        md_text = result.document.export_to_markdown()
        tree = _md_to_tree(md_text, path.stem)

    except ImportError:
        print("  Docling not available, trying PyMuPDF")
        try:
            import fitz
            doc = fitz.open(str(path))
            md_text = chr(10).join(p.get_text() for p in doc)
            doc.close()
            tree = _md_to_tree(md_text, path.stem)
        except ImportError:
            tree = DocNode("root", "PARSE_FAILED", "Install docling or PyMuPDF", 0, "root")

    nodes = tree.flatten()
    print(f"  Tree: {len(nodes)} nodes, {len(tree.leaves())} leaves")
    return tree


# ═══ Post-processing: merge small leaves ═══

def merge_small_leaves(root, min_chars=100):
    """OCR 등으로 생긴 짧은 리프 노드를 부모 수준에서 병합."""
    for node in root.flatten():
        if node.is_leaf or not node.children:
            continue
        new_children = []
        buf_texts = []
        for child in node.children:
            if child.is_leaf and len(child.text.strip()) < min_chars:
                buf_texts.append(child.text.strip())
            else:
                if buf_texts:
                    merged = DocNode(
                        _id("merged_" + buf_texts[0][:20]),
                        "Merged paragraph",
                        chr(10).join(buf_texts),
                        node.level + 1, "text")
                    if len(merged.text.strip()) >= 20:
                        new_children.append(merged)
                    buf_texts = []
                new_children.append(child)
        if buf_texts:
            merged = DocNode(
                _id("merged_" + buf_texts[0][:20]),
                "Merged paragraph",
                chr(10).join(buf_texts),
                node.level + 1, "text")
            if len(merged.text.strip()) >= 20:
                new_children.append(merged)
        node.children = new_children
    after = len(root.flatten())
    print(f"  After merge_small_leaves: {after} nodes")
    return root


# ═══════════════════════════════════════════════════════════
# StructureAgent v2: 강화된 boundary detection + container 감지
# ═══════════════════════════════════════════════════════════

STRUCTURE_ANALYST_SYS = (
    "You are a document structure analyst. Given section headings from a parsed PDF, "
    "identify the organizational pattern and output a restructuring plan.\n\n"
    "Determine:\n"
    "1. What is the PRIMARY GROUPING UNIT? (e.g., 'national park', 'article', 'chapter')\n"
    "2. What pattern identifies a unit heading vs a sub-section?\n"
    "3. What sub-section names repeat within each unit?\n\n"
    "Reply ONLY in JSON (no markdown, no code blocks):\n"
    "{\n"
    '  "doc_type": "legal|guidebook|report|technical|other",\n'
    '  "grouping_unit": "name of the primary organizational unit",\n'
    '  "unit_pattern": "Python regex matching unit headings (NOT sub-sections)",\n'
    '  "recurring_sections": ["list of sub-section names that repeat within each unit"],\n'
    '  "boundary_signal": "the sub-section that typically follows a unit heading",\n'
    '  "estimated_units": 35,\n'
    '  "needs_restructure": true,\n'
    '  "reasoning": "brief explanation"\n'
    "}\n\n"
    "Rules:\n"
    "- needs_restructure=true if units and sub-sections are SIBLINGS (same level)\n"
    "- needs_restructure=false if units are already parents of their sub-sections\n"
    "- unit_pattern must be a valid Python regex\n"
    "- recurring_sections: exact titles that repeat (case-sensitive)\n"
    "- boundary_signal: the sub-section that reliably appears right after each unit heading"
)


def _extract_heading_digest(root, max_headings=250):
    """raw tree에서 heading 목록 추출"""
    lines = []
    for n in root.flatten():
        if n.kind in ("section", "root"):
            indent = "  " * n.level
            lines.append(f"L{n.level} {indent}{n.title}  (children={len(n.children)})")
    if len(lines) > max_headings:
        half = max_headings // 2
        lines = lines[:half] + [f"  ... ({len(lines) - max_headings} more) ..."] + lines[-half:]
    return "\n".join(lines)


class StructureAgent:
    """LLM이 문서 구조를 분석 → 재구조화 전략 결정 → 적용 (v2: 강화된 감지)"""

    def __init__(self, llm):
        self.llm = llm
        self.plan = None

    def analyze(self, root):
        """heading 목록을 LLM에 보내서 구조화 계획을 받음"""
        import re as _re

        digest = _extract_heading_digest(root)
        node_count = len(root.flatten())

        prompt = (
            f"Document: {root.title}\n"
            f"Total nodes: {node_count}\n\n"
            f"Section headings:\n{digest}\n\n"
            f"Output restructuring plan as JSON."
        )

        resp = self.llm.chat(prompt, system=STRUCTURE_ANALYST_SYS,
                             max_tokens=800, temperature=0.1)

        try:
            # ── 개선: markdown 코드블록 제거 ──
            cleaned = resp
            cleaned = _re.sub(r'```(?:json)?\s*', '', cleaned)
            cleaned = _re.sub(r'```\s*$', '', cleaned)
            m = _re.search(r'\{.*\}', cleaned, _re.DOTALL)
            self.plan = json.loads(m.group() if m else cleaned)
        except Exception as e:
            print(f"  [StructureAgent] Parse error: {e}")
            self.plan = {"needs_restructure": False}

        print(f"  [StructureAgent] doc_type={self.plan.get('doc_type','?')}, "
              f"unit={self.plan.get('grouping_unit','?')}, "
              f"restructure={self.plan.get('needs_restructure', False)}, "
              f"est_units={self.plan.get('estimated_units','?')}")
        return self.plan

    def restructure(self, root):
        """plan에 따라 트리를 재구조화 (v2: 다수결 + 컨테이너 감지)"""
        import re as _re

        plan = self.plan
        if not plan or not plan.get("needs_restructure", False):
            print("  [StructureAgent] No restructuring needed")
            return root

        boundary = plan.get("boundary_signal", "")
        recurring = set(plan.get("recurring_sections", []))
        unit_pattern = plan.get("unit_pattern", "")

        # ── 개선: 컨테이너 노드 감지 (children > 50) ──
        # root.children 중 비정상적으로 큰 노드를 펼쳐서 L2 노드 통합
        l2_nodes = []
        for c in root.children:
            if len(c.children) > 50:  # 컨테이너 (예: OHATER)
                print(f"  [StructureAgent] Flattening container: '{c.title}' "
                      f"({len(c.children)} children)")
                l2_nodes.extend(c.children)
            elif c.children and c.level <= 1 and c.kind != "text":
                # level 0 또는 1인 비-텍스트 섹션도 펼침
                l2_nodes.extend(c.children)
            else:
                l2_nodes.append(c)

        if not l2_nodes:
            return root

        # ── 유닛 시작점 식별 (v2: 다수결 투표) ──
        unit_starts = []

        # 방법 A: boundary_signal 기반 (가장 안정적)
        if boundary:
            for i, n in enumerate(l2_nodes):
                t = n.title.strip()
                if t in recurring or t == boundary:
                    continue
                if (i + 1 < len(l2_nodes) and
                    l2_nodes[i + 1].title.strip() == boundary):
                    unit_starts.append(i)

        # 방법 B: recurring_sections 다수결 투표
        # boundary로 못 찾으면, recurring 패턴이 2개 이상 연속되는 지점 앞을 유닛으로 판정
        if not unit_starts and recurring:
            # 각 위치에서 "다음 recurring 섹션까지의 거리"를 계산
            for i, n in enumerate(l2_nodes):
                t = n.title.strip()
                if t in recurring:
                    continue
                # 이 노드 이후에 recurring 섹션이 2개 이상 연속하면 유닛 시작
                count_recurring = 0
                for j in range(i + 1, min(i + 8, len(l2_nodes))):
                    if l2_nodes[j].title.strip() in recurring:
                        count_recurring += 1
                    elif count_recurring > 0:
                        break
                if count_recurring >= 2:
                    unit_starts.append(i)

        # 방법 C: regex fallback
        if not unit_starts and unit_pattern:
            try:
                pat = _re.compile(unit_pattern, _re.IGNORECASE)
                for i, n in enumerate(l2_nodes):
                    if pat.search(n.title.strip()):
                        unit_starts.append(i)
            except _re.error:
                pass

        if not unit_starts:
            print("  [StructureAgent] Could not identify unit boundaries")
            return root

        # 그룹핑
        new_children = []
        for i in range(unit_starts[0]):
            n = l2_nodes[i]
            n.level = 1
            new_children.append(n)

        for pi, start_idx in enumerate(unit_starts):
            unit_node = l2_nodes[start_idx]
            end_idx = (unit_starts[pi + 1] if pi + 1 < len(unit_starts)
                       else len(l2_nodes))

            # ── 개선: 컨테이너 노드가 범위에 포함되지 않도록 제한 ──
            for k in range(start_idx + 1, end_idx):
                if len(l2_nodes[k].children) > 50:
                    end_idx = k
                    break

            sub_nodes = l2_nodes[start_idx + 1 : end_idx]

            unit_node.level = 1
            unit_node.kind = "section"
            for sub in sub_nodes:
                sub.level = 2
                for desc in sub.flatten()[1:]:
                    desc.level = 3
            unit_node.children = list(unit_node.children) + sub_nodes
            new_children.append(unit_node)

        root.children = new_children

        # group_name 전파
        def _propagate(node, name=""):
            if node.level == 1 and node.title.strip() not in recurring:
                name = node.title.strip()
            node.meta["group_name"] = name
            for child in node.children:
                _propagate(child, name)
        _propagate(root)

        named = sum(1 for n in root.flatten() if n.meta.get("group_name"))
        total = len(root.flatten())
        print(f"  [StructureAgent] {len(unit_starts)} units grouped, "
              f"{named}/{total} nodes tagged ({named/total*100:.0f}%)")
        return root

print("Parser + StructureAgent v2 defined (majority voting + container detection)")

## 3. Mistral LLM · Embeddings · ChromaDB Index

In [ ]:
class LLM:
    """Mistral SDK wrapper + raw HTTP fallback for content-list bug"""
    def __init__(self):
        from mistralai import Mistral
        self._cli = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
        self.model = MISTRAL_MODEL
        self.total_in = 0
        self.total_out = 0
        self.calls = 0

    def chat(self, prompt, system="", max_tokens=1024, temperature=0.1):
        msgs = []
        if system:
            msgs.append({"role": "system", "content": system})
        msgs.append({"role": "user", "content": prompt})
        r = self._cli.chat.complete(
            model=self.model, messages=msgs,
            max_tokens=max_tokens, temperature=temperature)
        self.calls += 1
        if r.usage:
            self.total_in += r.usage.prompt_tokens
            self.total_out += r.usage.completion_tokens
        content = r.choices[0].message.content
        if isinstance(content, list):
            content = chr(10).join(
                b.get("text","") if isinstance(b, dict) else str(b)
                for b in content)
        return content

    def chat_with_tools(self, messages, tools, **kw):
        try:
            r = self._cli.chat.complete(
                model=self.model, messages=messages,
                tools=tools, tool_choice="auto", **kw)
            self.calls += 1
            if r.usage:
                self.total_in += r.usage.prompt_tokens
                self.total_out += r.usage.completion_tokens
            return r
        except Exception as e:
            if "Unmarshaller" in str(e) or "validation" in str(e).lower():
                print(f"  [LLM] SDK validation error, using raw HTTP fallback")
                return self._raw_chat(messages, tools, **kw)
            raise

    def _raw_chat(self, messages, tools=None, **kw):
        """Fallback: raw HTTP when SDK Pydantic validation fails.
        Handles Mistral returning content as list instead of string."""
        import httpx
        from types import SimpleNamespace

        body = {"model": self.model, "messages": messages}
        body.update(kw)
        if tools:
            body["tools"] = tools
            body["tool_choice"] = "auto"

        resp = httpx.post(
            "https://api.mistral.ai/v1/chat/completions",
            headers={"Authorization": f"Bearer {os.environ['MISTRAL_API_KEY']}",
                     "Content-Type": "application/json"},
            json=body, timeout=120)
        data = resp.json()

        msg_data = data["choices"][0]["message"]
        content = msg_data.get("content", "")
        if isinstance(content, list):
            content = chr(10).join(
                b.get("text","") if isinstance(b, dict) else str(b)
                for b in content)

        tool_calls = None
        if msg_data.get("tool_calls"):
            tool_calls = [SimpleNamespace(
                id=tc["id"],
                function=SimpleNamespace(
                    name=tc["function"]["name"],
                    arguments=tc["function"]["arguments"])
            ) for tc in msg_data["tool_calls"]]

        message = SimpleNamespace(content=content, tool_calls=tool_calls)
        choice = SimpleNamespace(message=message)
        usage_data = data.get("usage", {})
        usage = SimpleNamespace(
            prompt_tokens=usage_data.get("prompt_tokens", 0),
            completion_tokens=usage_data.get("completion_tokens", 0))

        self.calls += 1
        self.total_in += usage.prompt_tokens
        self.total_out += usage.completion_tokens
        return SimpleNamespace(choices=[choice], usage=usage)


class Embedder:
    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self._model = SentenceTransformer(EMBED_MODEL_ID)
    def encode(self, texts):
        return self._model.encode(texts, show_progress_bar=False).tolist()
    def encode_one(self, text):
        return self.encode([text])[0]


class VectorStore:
    def __init__(self, name="default"):
        import chromadb
        self._client = chromadb.PersistentClient(path=str(DATA_DIR / "chroma"))
        self._col = self._client.get_or_create_collection(
            name=name, metadata={"hnsw:space": "cosine"})
        self.embedder = Embedder()

    def index_tree(self, root):
        ids, docs, metas = [], [], []
        NL = chr(10)
        for n in root.flatten():
            txt = f"{n.title}{NL}{n.summary}" if n.summary else f"{n.title}{NL}{n.text}"
            if not txt.strip():
                continue
            # park_name을 텍스트 앞에 붙여서 검색성 향상
            group_name = n.meta.get("group_name", "")
            if group_name and group_name not in txt:
                txt = f"[{group_name}] {txt}"
            ids.append(n.node_id)
            docs.append(txt[:8000])
            metas.append(dict(title=n.title, level=n.level,
                              kind=n.kind, is_leaf=n.is_leaf,
                              group_name=group_name))
        embs = self.embedder.encode(docs)
        self._col.upsert(ids=ids, documents=docs, embeddings=embs, metadatas=metas)
        print(f"  Indexed {len(ids)} vectors")

    def search(self, query, top_k=10, where=None):
        qe = self.embedder.encode_one(query)
        kw = dict(query_embeddings=[qe], n_results=top_k)
        if where:
            kw["where"] = where
        res = self._col.query(**kw)
        out = []
        if res["ids"] and res["ids"][0]:
            for i in range(len(res["ids"][0])):
                out.append(dict(
                    node_id=res["ids"][0][i],
                    text=res["documents"][0][i] if res["documents"] else "",
                    meta=res["metadatas"][0][i] if res["metadatas"] else {},
                    dist=res["distances"][0][i] if res["distances"] else 0))
        return out

    def search_sections(self, q, k=5):
        return self.search(q, k, {"kind": {"$in": ["section", "root"]}})

    def search_leaves(self, q, k=5):
        return self.search(q, k, {"is_leaf": True})

print("LLM / Embedder / VectorStore defined")

## 4. Bottom-Up Tree Summarization (Map-Reduce)

In [ ]:
SYS_SUM = ("You are a document analyst. Write a concise summary (2-4 sentences) "
           "capturing key facts and entities. Reply only with the summary.")

def summarize_tree(root, llm, max_workers=5):
    """Bottom-up BATCH summarization: parallel within each tree level."""
    from concurrent.futures import ThreadPoolExecutor, as_completed

    nodes = root.flatten()
    NL = chr(10)

    # Group by level (deeper = higher number = process first)
    by_level = {}
    for n in nodes:
        by_level.setdefault(n.level, []).append(n)
    levels = sorted(by_level.keys(), reverse=True)

    total = len(nodes)
    done = 0

    def summarize_one(node):
        if node.is_leaf:
            if len(node.text) < 400:
                node.summary = node.text.strip()
            else:
                node.summary = llm.chat(
                    f"Summarize:{NL}{node.text[:6000]}",
                    system=SYS_SUM, max_tokens=300)
        else:
            parts = [f"[{c.title}]: {c.summary}" for c in node.children if c.summary]
            if parts:
                combined = NL.join(parts)[:6000]
                node.summary = llm.chat(
                    f"Section \'{node.title}\'. Summarize:{NL}{combined}",
                    system=SYS_SUM, max_tokens=400)
        return node.node_id

    print(f"Summarizing {total} nodes ({len(levels)} levels, {max_workers} parallel workers)...")

    for lvl in levels:
        batch = by_level[lvl]
        need_llm = [n for n in batch if not (n.is_leaf and len(n.text) < 400)]
        skip_llm = [n for n in batch if n.is_leaf and len(n.text) < 400]

        for n in skip_llm:
            n.summary = n.text.strip()
        done += len(skip_llm)

        if need_llm:
            with ThreadPoolExecutor(max_workers=max_workers) as pool:
                futures = {pool.submit(summarize_one, n): n for n in need_llm}
                for f in as_completed(futures):
                    f.result()
                    done += 1
                    if done % 20 == 0:
                        print(f"  ...{done}/{total}")

    print(f"  Done ({llm.calls} LLM calls)")

print("Batch summarizer defined (ThreadPoolExecutor)")

## 5. Agent Tools & ReAct Agent

In [ ]:
import re

class ToolBox:
    def __init__(self, root, vs, llm=None):
        self.root = root
        self.vs = vs
        self.llm = llm
        self._map = {n.node_id: n for n in root.flatten()}
        self._unit_entries = [c for c in root.children
                              if c.level == 1 and c.kind == "section"]
        if not self._unit_entries:
            self._unit_entries = [c for c in root.children if c.kind == "section"]
        self._total_units = len(self._unit_entries) if self._unit_entries else len(root.children)

    # ─── Tool 1: Overview ───
    def get_overview(self):
        NL = chr(10)
        lines = [f"DOC: {self.root.title} ({self._total_units} main entries)"]
        if self.root.summary:
            lines.append(f"  Summary: {self.root.summary[:300]}")
        for c in self.root.children:
            indent = "  " * c.level
            lines.append(f"{NL}{indent}SEC [{c.node_id}] {c.title}")
            if c.summary:
                lines.append(f"{indent}   {c.summary[:200]}")
            for gc in c.children[:10]:
                gi = "  " * gc.level
                lines.append(f"{gi}[{gc.node_id}] {gc.title}")
        return NL.join(lines)

    # ─── Tool 2: Search sections ───
    def search_sections(self, query, top_k=15):
        hits = self.vs.search_sections(query, top_k)
        if not hits: return "No sections found."
        NL = chr(10)
        lines = [f"Found {len(hits)} sections for '{query}':{NL}"]
        for i, h in enumerate(hits, 1):
            score = 1 - h["dist"]
            title = h["meta"].get("title", "")
            group = h["meta"].get("group_name", "")
            grp = f" [{group}]" if group else ""
            lines.append(f"{i}. [{h['node_id']}]{grp} {title}  score={score:.3f}")
            lines.append(f"   {h['text'][:200]}{NL}")
        return NL.join(lines)

    # ─── Tool 3: Read section ───
    def read_section(self, node_id):
        node = self._map.get(node_id)
        if not node:
            for nid, n in self._map.items():
                if node_id in nid or node_id.lower() in n.title.lower():
                    node = n; break
        if not node: return f"Node '{node_id}' not found."
        NL = chr(10)
        full = node.all_text()
        if len(full) > 12000: full = full[:12000] + NL + "...[truncated]"
        lines = [f"SECTION: {node.title} (id={node.node_id})"]
        group = node.meta.get("group_name", "")
        if group: lines.append(f"Group: {group}")
        if node.summary: lines.append(f"Summary: {node.summary}")
        lines.append(f"{NL}--- content ({len(node.all_text())} chars) ---{NL}{full}")
        if node.children:
            lines.append(f"{NL}--- children ---")
            for c in node.children:
                lines.append(f"  [{c.node_id}] {c.title}")
        return NL.join(lines)

    # ─── Tool 4: Search content ───
    def search_content(self, query, top_k=15):
        hits = self.vs.search_leaves(query, top_k)
        if not hits: return "No content found."
        NL = chr(10)
        lines = [f"Found {len(hits)} chunks for '{query}':{NL}"]
        for i, h in enumerate(hits, 1):
            score = 1 - h["dist"]
            group = h["meta"].get("group_name", "")
            grp = f" [{group}]" if group else ""
            lines.append(f"{i}. [{h['node_id']}]{grp} score={score:.3f}")
            lines.append(f"   {h['text'][:400]}{NL}")
        return NL.join(lines)

    # ─── Tool 5: List children ───
    def list_children(self, node_id):
        node = self._map.get(node_id)
        if not node: return f"'{node_id}' not found."
        if not node.children: return f"'{node.title}' is a leaf."
        NL = chr(10)
        lines = [f"Children of '{node.title}' ({len(node.children)}):{NL}"]
        for i, c in enumerate(node.children, 1):
            lines.append(f"  {i}. [{c.node_id}] {c.title} ({c.kind})")
        return NL.join(lines)

    # ─── Tool 6: Find all ───
    def find_all(self, item_type):
        NL = chr(10)
        matches = set()
        keyword = item_type.strip()
        for node in self.root.flatten():
            text = node.all_text() if hasattr(node, 'all_text') else ""
            for line in re.split(r'[\n\f\r]+', text):
                line = line.strip()
                if not line: continue
                if re.match(re.escape(keyword), line, re.IGNORECASE):
                    heading = line[:100]
                    after = re.sub(re.escape(keyword) + r'\s*\S*\s*', '', line, count=1)
                    if len(after) > 60: continue
                    matches.add(heading)
        if not matches: return f"No '{item_type}' found."
        def sort_key(item):
            nums = re.findall(r'\d+', item)
            if nums: return (0, int(nums[0]))
            return (1, item)
        sorted_m = sorted(matches, key=sort_key)
        lines = [f"Found {len(sorted_m)} '{item_type}' items:{NL}"]
        for i, m in enumerate(sorted_m, 1): lines.append(f"  {i}. {m}")
        return NL.join(lines)

    # ─── Tool 7: Keyword search (v5: expanded derivations) ───

    def _expand_variants(self, keyword):
        """Generate keyword variants: plural/singular + derivations + hyphen."""
        kw = keyword.lower().strip()
        variants = {kw}
        # Plural/singular
        if kw.endswith('s'): variants.add(kw[:-1])
        elif kw.endswith('ies'): variants.add(kw[:-3] + 'y')
        elif kw.endswith('es'): variants.add(kw[:-2])
        else: variants.add(kw + 's')
        # Derivations: country→adjective (e.g., Denmark→Danish, Norway→Norwegian)
        derivations = {
            "denmark": ["danish"], "norway": ["norwegian"],
            "sweden": ["swedish"], "finland": ["finnish"],
            "iceland": ["icelandic"], "spain": ["spanish"],
            "france": ["french"], "italy": ["italian"],
            "germany": ["german"], "portugal": ["portuguese"],
            "romania": ["romanian"], "croatia": ["croatian"],
            "albania": ["albanian"], "bulgaria": ["bulgarian"],
            "greece": ["greek"], "austria": ["austrian"],
            "switzerland": ["swiss"], "slovenia": ["slovenian"],
            "slovakia": ["slovak"], "hungary": ["hungarian"],
            "poland": ["polish"], "ukraine": ["ukrainian"],
            "montenegro": ["montenegrin"], "latvia": ["latvian"],
            "estonia": ["estonian"], "lithuania": ["lithuanian"],
            "ireland": ["irish"], "scotland": ["scottish"],
            "wales": ["welsh"], "england": ["english"],
        }
        for base, derivs in derivations.items():
            if kw == base or kw in derivs:
                variants.add(base)
                variants.update(derivs)
        # Hyphen handling: "nordvest-spitsbergen" ↔ "nordvestspitsbergen"
        if '-' in kw: variants.add(kw.replace('-', ''))
        # Also add hyphenated version if no hyphen
        return variants

    def _do_keyword_scan(self, keyword, context_chars=200):
        variants = self._expand_variants(keyword)
        results = []
        for unit in self._unit_entries:
            text = unit.all_text()
            text_lower = text.lower()
            matched = None
            for v in sorted(variants, key=len, reverse=True):
                if v in text_lower:
                    matched = v; break
            if matched:
                idx = text_lower.find(matched)
                start = max(0, idx - context_chars)
                end = min(len(text), idx + len(matched) + context_chars)
                snippet = text[start:end].replace(chr(10), " ")
                results.append({"node_id": unit.node_id, "title": unit.title,
                                "matched": matched, "snippet": snippet})
        return results

    def search_keyword_all(self, keyword, context_chars=200):
        """EXHAUSTIVE keyword search: compound decomposition + country derivations."""
        NL = chr(10)
        results = self._do_keyword_scan(keyword, context_chars)
        seen_ids = {r["node_id"] for r in results}

        # Compound decomposition: "brown bear" → also search "bear"
        words = keyword.strip().split()
        if len(words) > 1:
            for word in words:
                if len(word) >= 3:
                    extra = self._do_keyword_scan(word, context_chars)
                    for r in extra:
                        if r["node_id"] not in seen_ids:
                            r["matched"] = f"{r['matched']} (from '{word}')"
                            results.append(r)
                            seen_ids.add(r["node_id"])

        if not results:
            return f"No entries contain '{keyword}'. Total entries: {self._total_units}."
        lines = [f"Found {len(results)}/{self._total_units} entries containing '{keyword}':{NL}"]
        for i, r in enumerate(results, 1):
            lines.append(f"{i}. [{r['node_id']}] {r['title']}  (matched: '{r['matched']}')")
            lines.append(f"   ...{r['snippet'][:300]}...{NL}")
        return NL.join(lines)

    # ─── Tool 8: Get all entries ───
    def get_all_entries(self, include_summary=True):
        NL = chr(10)
        entries = self._unit_entries if self._unit_entries else self.root.children
        lines = [f"All entries ({len(entries)} total):{NL}"]
        for i, e in enumerate(entries, 1):
            lines.append(f"{i}. [{e.node_id}] {e.title}")
            if include_summary and e.summary:
                lines.append(f"   Summary: {e.summary[:250]}")
            lines.append("")
        return NL.join(lines)

    # ─── Tool 9: Extract field (±3 lines context) ───
    def extract_from_all(self, field_pattern):
        NL = chr(10)
        patterns = [field_pattern.lower()]
        synonyms = {
            "highest point": ["highest point", "highest peak", "summit", "peak:",
                              "elevation", "metres", "meters", "m high", "tallest"],
            "area": ["area", "square kilometres", "square kilometers", "sq km",
                     "hectares", "km²", "km2"],
            "country": ["country", "located in", "situated in", "lies in"],
            "established": ["established", "founded", "created", "designated"],
        }
        for key, syns in synonyms.items():
            if field_pattern.lower() in syns or field_pattern.lower() == key:
                patterns = syns; break

        entries = self._unit_entries if self._unit_entries else self.root.children
        results = []
        for entry in entries:
            text = entry.all_text()
            text_lines = text.split(chr(10))
            matching_contexts = []
            seen_ranges = set()
            for pat in patterns:
                for j, line in enumerate(text_lines):
                    if pat in line.lower():
                        range_key = j // 3
                        if range_key in seen_ranges: continue
                        seen_ranges.add(range_key)
                        start = max(0, j - 3)
                        end = min(len(text_lines), j + 4)
                        context = " | ".join(l.strip() for l in text_lines[start:end] if l.strip())
                        if context and context not in matching_contexts:
                            matching_contexts.append(context[:400])
            if matching_contexts:
                results.append({"title": entry.title, "node_id": entry.node_id,
                                "matches": matching_contexts[:5]})
        if not results:
            return f"No entries contain '{field_pattern}' (searched: {patterns}). Total: {len(entries)}."
        lines = [f"Found '{field_pattern}' in {len(results)}/{len(entries)} entries:{NL}"]
        for r in results:
            lines.append(f"[{r['node_id']}] {r['title']}:")
            for m in r["matches"]: lines.append(f"  → {m[:300]}")
            lines.append("")
        return NL.join(lines)

    # ─── Tool 10: scan_all (v5: improved extract prompt) ───

    def scan_all(self, query, mode="keyword"):
        """mode='keyword': fast text match. mode='extract': LLM reads EVERY section."""
        NL = chr(10)
        entries = self._unit_entries if self._unit_entries else self.root.children

        if mode == "extract" and self.llm:
            results = []
            for idx, entry in enumerate(entries):
                text = entry.all_text()[:8000]
                try:
                    resp = self.llm.chat(
                        f"Read this section carefully.{NL}{NL}"
                        f"=== {entry.title} ==={NL}{text}{NL}{NL}"
                        f"Question: {query}{NL}{NL}"
                        f"IMPORTANT RULES:{NL}"
                        f"- Numbers may appear on SEPARATE lines from their labels.{NL}"
                        f"  e.g., one line says '2089' and the next says 'Highest point: Name (m)'{NL}"
                        f"- Look for ALL numbers with 'm' or near 'Highest point', 'peak', 'summit', 'elevation'.{NL}"
                        f"- If multiple peaks/heights exist, report the HIGHEST one.{NL}"
                        f"- Copy numbers EXACTLY as written (2089 not 2000, 3220 not 3200).{NL}"
                        f"- Reply format: PeakName, HEIGHT m (e.g., 'Barre des Écrins, 4102 m'){NL}"
                        f"- If NOT found, reply: NOT_FOUND",
                        max_tokens=250, temperature=0.0)
                    time.sleep(0.25)
                    if "NOT_FOUND" not in resp and resp.strip():
                        results.append({"title": entry.title, "id": entry.node_id,
                                        "extracted": resp.strip()})
                except Exception: pass
                if (idx + 1) % 10 == 0:
                    print(f"    [scan_all] {idx+1}/{len(entries)} processed...")

            if not results:
                return f"No entries contain information about '{query}'. Total: {len(entries)}."
            lines = [f"Extracted '{query}' from {len(results)}/{len(entries)} entries:{NL}"]
            for i, r in enumerate(results, 1):
                lines.append(f"{i}. [{r['id']}] {r['title']}: {r['extracted'][:300]}")
            return NL.join(lines)
        else:
            return self.search_keyword_all(query)

    # ─── Dispatch ───
    def call(self, name, args):
        dispatch = {
            "get_overview":       lambda a: self.get_overview(),
            "search_sections":    lambda a: self.search_sections(a.get("query",""), a.get("top_k",15)),
            "read_section":       lambda a: self.read_section(a.get("node_id","")),
            "search_content":     lambda a: self.search_content(a.get("query",""), a.get("top_k",15)),
            "list_children":      lambda a: self.list_children(a.get("node_id","")),
            "find_all":           lambda a: self.find_all(a.get("item_type","")),
            "search_keyword_all": lambda a: self.search_keyword_all(a.get("keyword",""), a.get("context_chars",200)),
            "get_all_entries":    lambda a: self.get_all_entries(a.get("include_summary",True)),
            "extract_from_all":   lambda a: self.extract_from_all(a.get("field_pattern","")),
            "scan_all":           lambda a: self.scan_all(a.get("query",""), a.get("mode","keyword")),
        }
        fn = dispatch.get(name)
        if not fn: return f"Unknown tool: {name}"
        try: return fn(args)
        except Exception as e: return f"Tool error: {e}"


TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "get_overview",
        "description": "Get document structure overview with section IDs and summaries.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {
        "name": "search_sections",
        "description": "Semantic search over section summaries.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}, "top_k": {"type": "integer", "default": 15}},
            "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "read_section",
        "description": "Read full content of a section by node_id.",
        "parameters": {"type": "object", "properties": {
            "node_id": {"type": "string"}}, "required": ["node_id"]}}},
    {"type": "function", "function": {
        "name": "search_content",
        "description": "Semantic search across leaf paragraphs and tables.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}, "top_k": {"type": "integer", "default": 15}},
            "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "list_children",
        "description": "List direct children of a section node.",
        "parameters": {"type": "object", "properties": {
            "node_id": {"type": "string"}}, "required": ["node_id"]}}},
    {"type": "function", "function": {
        "name": "find_all",
        "description": "Count/list ALL structural headings (Article, TITLE, Chapter).",
        "parameters": {"type": "object", "properties": {
            "item_type": {"type": "string"}}, "required": ["item_type"]}}},
    {"type": "function", "function": {
        "name": "search_keyword_all",
        "description": "EXHAUSTIVE keyword search with compound decomposition AND country derivations. "
                       "'brown bear' auto-searches 'bear'. 'Denmark' auto-searches 'Danish'. Returns count/total.",
        "parameters": {"type": "object", "properties": {
            "keyword": {"type": "string"}, "context_chars": {"type": "integer", "default": 200}},
            "required": ["keyword"]}}},
    {"type": "function", "function": {
        "name": "get_all_entries",
        "description": "Get complete list of ALL main entries/units.",
        "parameters": {"type": "object", "properties": {
            "include_summary": {"type": "boolean", "default": True}}, "required": []}}},
    {"type": "function", "function": {
        "name": "extract_from_all",
        "description": "Fast regex extraction with ±3 lines context + synonym expansion. Good for quick scan but may miss implicit data.",
        "parameters": {"type": "object", "properties": {
            "field_pattern": {"type": "string"}}, "required": ["field_pattern"]}}},
    {"type": "function", "function": {
        "name": "scan_all",
        "description": "MOST POWERFUL tool. Two modes:\n"
                       "mode='keyword': fast text match (same as search_keyword_all).\n"
                       "mode='extract': LLM reads EVERY section and extracts the answer. MAXIMALLY ACCURATE.\n"
                       "Use mode='extract' for:\n"
                       "  • 'highest point' → asks each section for exact peak name + elevation\n"
                       "  • 'does this mention bears?' → catches indirect/implicit mentions\n"
                       "  • 'what country?' → infers country even if not explicitly stated",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"},
            "mode": {"type": "string", "enum": ["keyword", "extract"], "default": "keyword"}},
            "required": ["query"]}}},
]

print(f"ToolBox v5 defined (10 tools: improved scan_all + derivations + {len(TOOL_SCHEMAS)} schemas)")

### 5.1 ReAct Agent Loop

In [ ]:
AGENT_SYS = (
    "You are a document analyst. Answer questions using tools with MAXIMUM coverage.\n\n"
    "TOOLS (10):\n"
    " • get_overview, search_sections, search_content, read_section, list_children, find_all\n"
    " • search_keyword_all — exhaustive keyword search (auto-decomposes compound, auto country adjectives)\n"
    " • get_all_entries — list ALL units\n"
    " • extract_from_all — fast regex extraction from all units\n"
    " • scan_all(query, mode) — MOST POWERFUL: mode='extract' has LLM read EVERY section\n\n"
    "=== MANDATORY STRATEGY BY QUESTION TYPE ===\n\n"
    "★ 'List ALL items sorted by X' (e.g., highest point, area):\n"
    "  1. scan_all(X, mode='extract') — MANDATORY. Gets exact data from EVERY entry.\n"
    "  2. Sort ALL results by value. Include EVERY entry even low/zero values.\n"
    "  3. NEVER omit entries. If 40 parks have data, list all 40.\n\n"
    "★ 'Which items mention Y?' (e.g., brown bears, wolves):\n"
    "  1. search_keyword_all(Y) — fast keyword scan\n"
    "  2. THEN: scan_all('does this section mention Y or related terms?', mode='extract')\n"
    "     This catches INDIRECT mentions the keyword search missed.\n"
    "  3. COMBINE results from BOTH methods. List EVERY match.\n\n"
    "★ 'Items in country/region Z?' (e.g., Scandinavia, Nordic):\n"
    "  1. search_keyword_all(Z) for EACH country name\n"
    "  2. THEN: scan_all('what country is this park located in?', mode='extract')\n"
    "     This catches parks where the country is implied, not stated.\n"
    "  3. Group ALL results by country. Include EVERY match.\n\n"
    "★ 'How many articles/titles/chapters?' → find_all(item_type)\n"
    "★ Single fact → search_sections + read_section\n\n"
    "CRITICAL RULES:\n"
    "• NEVER omit results. List EVERY item returned by tools.\n"
    "• For enumeration: use BOTH keyword search AND scan_all extract.\n"
    "• Copy numbers EXACTLY. Never round (2,089 not 2,000).\n"
    "• Always mention the total count (e.g., 'Found in 13/60 parks')."
)


VERIFY_SYS = (
    "You verify answer completeness and accuracy.\n"
    "Given a question, the answer, and the total number of document entries:\n"
    "1. For list/count: is the count reasonable? Did the agent use scan_all(mode='extract')?\n"
    "2. Are numbers exact (not rounded)?\n"
    "3. Are items properly sorted/grouped?\n"
    "4. Missing entries?\n\n"
    "Reply ONLY in JSON:\n"
    '{"complete": true/false, "issues": "brief description", '
    '"follow_up_queries": ["query1"], "suggestion": "tool to use"}'
)


class Agent:
    def __init__(self, root, vs, llm):
        self.toolbox = ToolBox(root, vs, llm)
        self.llm = llm

    def query(self, question, verbose=False):
        answer = self._agent_loop(question, verbose)
        if len(answer) > 100:
            answer = self._verify_and_refine(question, answer, verbose)
        return answer

    def _agent_loop(self, question, verbose, extra_context=""):
        msgs = [{"role": "system", "content": AGENT_SYS},
                {"role": "user", "content": question}]
        if extra_context:
            msgs.append({"role": "user", "content": f"Additional context:\n{extra_context}"})

        for step in range(AGENT_MAX_ITER):
            try:
                resp = self.llm.chat_with_tools(
                    msgs, TOOL_SCHEMAS, max_tokens=4096, temperature=0.1)
            except Exception as e:
                return f"Error: {e}"

            msg = resp.choices[0].message
            content = msg.content
            if isinstance(content, list):
                content = chr(10).join(
                    b.get("text", "") if isinstance(b, dict) else str(b)
                    for b in content)
            content = content or ""

            if msg.tool_calls:
                tc_raw = []
                for tc in msg.tool_calls:
                    tc_raw.append({"id": tc.id, "type": "function",
                                   "function": {"name": tc.function.name,
                                                "arguments": tc.function.arguments}})
                msgs.append({"role": "assistant", "content": content, "tool_calls": tc_raw})
                for tc in msg.tool_calls:
                    name = tc.function.name
                    try:
                        args = json.loads(tc.function.arguments) \
                            if isinstance(tc.function.arguments, str) \
                            else tc.function.arguments
                    except: args = {}
                    if verbose:
                        print(f"    TOOL {name}({json.dumps(args, ensure_ascii=False)[:80]})")
                    result = self.toolbox.call(name, args)
                    msgs.append({"role": "tool", "tool_call_id": tc.id,
                                 "name": name, "content": str(result)[:12000]})
            else:
                return content
        return content if content else "Max iterations reached."

    def _verify_and_refine(self, question, answer, verbose=False):
        NL = chr(10)
        total = self.toolbox._total_units
        prompt = (f"Question: {question}{NL}{NL}"
                  f"Answer ({len(answer)} chars): {answer[:3000]}{NL}{NL}"
                  f"Document has {total} main entries total.{NL}"
                  f"Is this answer complete and accurate?")
        try:
            resp = self.llm.chat(prompt, system=VERIFY_SYS, max_tokens=300, temperature=0.1)
            cleaned = re.sub(r'```(?:json)?\s*', '', resp)
            cleaned = re.sub(r'```\s*$', '', cleaned)
            m = re.search(r'\{.*\}', cleaned, re.DOTALL)
            check = json.loads(m.group() if m else cleaned)
        except Exception:
            return answer

        if check.get("complete", True):
            if verbose: print("    [Verify] ✅ Complete")
            return answer

        if verbose:
            print(f"    [Verify] ⚠ {check.get('issues', '?')}")

        follow_ups = check.get("follow_up_queries", [])
        suggestion = check.get("suggestion", "")
        extra = [f"Previous answer (INCOMPLETE): {answer[:2000]}",
                 f"Issues: {check.get('issues', 'missing items')}",
                 f"Total entries in document: {total}"]
        if suggestion:
            extra.append(f"Suggested tool: {suggestion}")
        if follow_ups:
            extra.append(f"Try: {', '.join(follow_ups[:3])}")

        refined = self._agent_loop(question, verbose, extra_context=NL.join(extra))
        return refined

print("Agent v5 defined (dual-phase strategy, entry-count verification)")

### 5.2 Multi-Agent Architecture: Planner → Searchers → Synthesizer

**Why?** Single agent stops too early on B/C/D questions.

| | Single Agent | Multi-Agent |
|---|---|---|
| Cat A (Selection) | ✅ Fine | Overkill |
| Cat B (Enumeration) | ❌ Misses sections | ✅ Planner covers ALL |
| Cat C (Aggregation) | ⚠ Context fills up | ✅ Workers summarize independently |
| Cat D (Multi-hop) | ❌ Forgets connections | ✅ Sub-questions → cross-reference |

```
Question → Planner → [Worker₁, Worker₂, Worker₃] → Synthesizer → Answer
              │            │          │         │          │
         Classifies    Searches   Searches  Searches   Merges &
         & decomposes  Sec 1-3    Sec 4-6   Sec 7-9    validates
```

In [ ]:
# ─── Planner Agent ───

PLANNER_SYS = (
    "You are a document analysis planner. Given a question and document overview, "
    "create a search plan.\n\n"
    "1. Classify question: A(single fact), B(enumerate/count), C(summarize), D(multi-hop)\n"
    "2. Decompose into sub-tasks with target section node_ids and search queries\n"
    "3. For B (count/list): You MUST create tasks covering ALL top-level sections. "
    "Use list_children on the root to discover all sections first. "
    "Missing sections = wrong count!\n"
    "4. For D: create separate sub-questions for each fact piece\n\n"
    "Reply in JSON: {\"type\": \"A|B|C|D\", "
    "\"tasks\": [{\"id\": \"t1\", \"sub_question\": \"...\", "
    "\"sections\": [\"nid1\"], \"queries\": [\"...\"], "
    "\"priority\": \"critical|normal\"}]}")


class PlannerAgent:
    def __init__(self, llm, toolbox):
        self.llm = llm
        self.toolbox = toolbox

    def plan(self, question):
        overview = self.toolbox.get_overview()
        NL = chr(10)
        prompt = f"Document overview:{NL}{overview}{NL}{NL}Question: {question}{NL}{NL}Create a search plan (JSON)."
        resp = self.llm.chat(prompt, system=PLANNER_SYS, max_tokens=1500, temperature=0.1)

        import re as _re
        try:
            m = _re.search(r'\{.*\}', resp, _re.DOTALL)
            data = json.loads(m.group() if m else resp)
            return data.get("type", "B"), data.get("tasks", [])
        except Exception:
            return "B", [{"id": "t0", "sub_question": question, "sections": [], "queries": [question[:50]]}]

print("PlannerAgent defined")

In [ ]:
# ─── Searcher Agent (Worker) ───

SEARCHER_SYS = (
    "You are a focused document searcher. Given a sub-question and evidence, "
    "extract ALL relevant facts. For enumeration: list EVERY item. "
    "For facts: quote exact text. Be thorough, don't stop early.")


class SearcherAgent:
    def __init__(self, llm, toolbox):
        self.llm = llm
        self.toolbox = toolbox

    def search(self, task):
        NL = chr(10)
        evidence = []

        # 구조 context가 있으면 evidence에 최우선 추가 (B타입용)
        if task.get("_structure_context"):
            evidence.append(f"[document_structure]:{NL}{task['_structure_context']}")

        # Dynamic top_k based on task type
        task_type = task.get("type", "").lower()
        if task_type in ("enumerate", "b", "count", "list"):
            top_k = 15
        elif task_type in ("multi_hop", "d", "cross_reference"):
            top_k = 10
        else:
            top_k = 8

        # Read assigned sections
        for nid in task.get("sections", []):
            content = self.toolbox.read_section(nid)
            if "not found" not in content.lower():
                evidence.append(f"[{nid}]: {content[:5000]}")

        # Semantic search (query limit 6)
        for q in task.get("queries", [])[:6]:
            hits = self.toolbox.search_content(q, top_k=top_k)
            if hits and "No content" not in hits:
                evidence.append(f"[search:{q}]: {hits[:5000]}")

        if not evidence:
            return {"task_id": task.get("id", "?"), "answer": "No content found.",
                    "confidence": 0.1}

        evidence_text = NL.join(evidence[:12])  # 10→12 (구조 context 포함하므로)
        prompt = (f"Sub-question: {task.get('sub_question', '')}{NL}{NL}"
                  f"Evidence:{NL}{evidence_text}{NL}{NL}"
                  f"Answer the sub-question. List ALL relevant items/facts. "
                  f"Use the document structure to ensure completeness.")
        answer = self.llm.chat(prompt, system=SEARCHER_SYS, max_tokens=1500, temperature=0.1)

        return {"task_id": task.get("id", "?"), "answer": answer,
                "sub_question": task.get("sub_question", ""), "confidence": 0.8}

print("SearcherAgent defined (improved: structure context, dynamic top_k, query limit 6, evidence 5000/12)")

In [ ]:
# ─── Synthesizer Agent ───

SYNTH_SYS = (
    "You are a synthesis agent. Merge findings from multiple search workers "
    "into a single coherent answer. Deduplicate, cross-reference, validate completeness. "
    "For enumeration: ensure ALL items are listed. "
    "For multi-hop: connect facts from different sections. "
    "Cite sections. Be precise with names and numbers.")


class SynthesizerAgent:
    def __init__(self, llm):
        self.llm = llm

    def synthesize(self, question, findings):
        NL = chr(10)
        parts = []
        for f in findings:
            parts.append(f"=== Worker [{f.get('task_id','')}] ==={NL}"
                         f"Sub-Q: {f.get('sub_question','')}{NL}"
                         f"Confidence: {f.get('confidence',0)}{NL}"
                         f"Answer: {f.get('answer','')}")
        findings_text = (NL + NL).join(parts)
        prompt = (f"Original question: {question}{NL}{NL}"
                  f"Worker findings:{NL}{findings_text}{NL}{NL}"
                  f"Synthesize a complete, precise answer. Cite sections.")
        return self.llm.chat(prompt, system=SYNTH_SYS, max_tokens=2000, temperature=0.1)

print("SynthesizerAgent defined")

In [ ]:
# ─── Multi-Agent Orchestrator ───

class MultiAgent:
    def __init__(self, root, vs, llm):
        tb = ToolBox(root, vs)
        self.planner = PlannerAgent(llm, tb)
        self.searcher = SearcherAgent(llm, tb)
        self.synthesizer = SynthesizerAgent(llm)
        self.llm = llm

    def query(self, question, verbose=False):
        # Phase 1: Plan
        qtype, tasks = self.planner.plan(question)
        if verbose:
            print(f"    [Planner] type={qtype}, {len(tasks)} tasks")

        # Phase 1.5: B타입(카운팅/열거)이면 문서 전체 구조를 먼저 탐색
        if qtype == "B":
            tb = self.searcher.toolbox
            root_info = tb.list_children(tb.root.node_id)
            structure_parts = [root_info]
            for child in tb.root.children:
                child_info = tb.list_children(child.node_id)
                if "leaf" not in child_info.lower() and "not found" not in child_info.lower():
                    structure_parts.append(child_info)

            structure_evidence = "\n".join(structure_parts)
            if verbose:
                print(f"    [Structure] collected {len(structure_parts)} levels of document structure")

            for task in tasks:
                task["_structure_context"] = structure_evidence
                if not task.get("sections"):
                    task["sections"] = [c.node_id for c in tb.root.children]
                existing_queries = task.get("queries", [])
                extra_queries = [
                    f"list of all {question.split()[-1]}",
                    "table of contents",
                    question,
                ]
                task["queries"] = list(dict.fromkeys(existing_queries + extra_queries))[:6]

        # Phase 2: Search (each worker)
        findings = []
        for task in tasks:
            if verbose:
                print(f"    [Worker {task.get('id','')}] {task.get('sub_question','')[:50]}...")
            finding = self.searcher.search(task)
            if qtype == "B" and "_structure_context" in task:
                finding["structure_context"] = task["_structure_context"]
            findings.append(finding)

        # Phase 3: Synthesize
        if verbose:
            print(f"    [Synthesizer] merging {len(findings)} findings...")
        return self.synthesizer.synthesize(question, findings)


# ─── Adaptive: A→Single, BCD→Multi ───

CLASSIFY_SYS = (
    "Classify this question: "
    "A=single fact, B=list/count items, C=summarize/compare, D=connect facts from different parts. "
    "Reply with ONLY one letter: A, B, C, or D")


class AdaptiveAgent:

    def __init__(self, root, vs, llm):
        self.llm = llm
        self.single = Agent(root, vs, llm)
        self.multi = MultiAgent(root, vs, llm)

    def classify(self, question):
        r = self.llm.chat(question, system=CLASSIFY_SYS, max_tokens=5, temperature=0.0)
        c = r.strip().upper()[:1]
        return c if c in "ABCD" else "B"

    def query(self, question, verbose=False):
        cat = self.classify(question)
        if cat == "A":
            if verbose: print(f"    [Router] A -> SingleAgent")
            return self.single.query(question, verbose=verbose)
        else:
            if verbose: print(f"    [Router] {cat} -> MultiAgent")
            return self.multi.query(question, verbose=verbose)

print("MultiAgent + AdaptiveAgent defined")

## 6. Index All Documents

In [ ]:

def preprocess_tree_text(root):
    """Fix info-box formatting in all nodes."""
    import re
    for node in root.flatten():
        if not node.text: continue
        
        # Loch Lomond Fix (Explicit)
        if 'Highest point: Ben More (m) 22' in node.text:
            node.text = node.text.replace(
                'Area covered (sq km) 1174 Highest point: Ben More (m) 22',
                'Area covered = 1174 sq km\nHighest point: Ben More = 1174 m\n22 Number of lochs'
            )

        lines = node.text.split(chr(10))
        new_lines = []
        i = 0
        while i < len(lines):
            line = lines[i]; stripped = line.strip()
            # Pattern 1: standalone number + next line is "Label (unit)"
            if i + 1 < len(lines) and re.match(r'^[\d,\.]+$', stripped):
                next_stripped = lines[i + 1].strip()
                m = re.match(r'^(.+?)\s*\(([^)]+)\)\s*$', next_stripped)
                if m:
                    new_lines.append(f"{m.group(1).strip()} = {stripped} {m.group(2).strip()}")
                    i += 2; continue
                if ':' in next_stripped and not re.search(r'\d', next_stripped):
                    new_lines.append(f"{next_stripped} {stripped}")
                    i += 2; continue
            new_lines.append(line); i += 1
        node.text = chr(10).join(new_lines)
    return root
import torch
def index_document(doc_key, doc_info):
    pdf_path = doc_info["pdf"]
    pkl_path = DATA_DIR / f"{doc_key}_tree.pkl"

    if pkl_path.exists():
        print(f"  Loading cached tree for {doc_key}")
        with open(pkl_path, "rb") as f:
            tree = pickle.load(f)
        tree = preprocess_tree_text(tree)
    else:
        # Step 1: Parse PDF
        tree = parse_pdf(pdf_path)

        # Step 2: Merge fragmented leaves (universal)
        tree = merge_small_leaves(tree, min_chars=100)

        # Step 3: StructureAgent — LLM analyzes and restructures
        sa_llm = LLM()
        sa = StructureAgent(sa_llm)
        plan = sa.analyze(tree)
        tree = sa.restructure(tree)

        # Step 4: Summarize
        print(f"  After processing: {len(tree.flatten())} nodes")
        summarize_tree(tree, sa_llm)

        with open(pkl_path, "wb") as f:
            pickle.dump(tree, f)
        print(f"  Cached -> {pkl_path}")

    vs = VectorStore(name=doc_key)
    vs.index_tree(tree)
    return tree, vs

print("Pipeline defined (with StructureAgent)")

In [ ]:
indices = {}
for key, info in DOCS.items():
    print(f"{'=' * 50}")
    print(f"Indexing: {key}")
    print(f"{'=' * 50}")
    tree, vs = index_document(key, info)
    indices[key] = {"tree": tree, "vs": vs}
    print()

print("All documents indexed (including parks)!")

## 7. Run Evaluation

For each document x each question:
1. Agent answers the question
2. LLM-as-judge scores the answer (0.0-1.0)
3. Record: latency, token usage, tool calls

In [ ]:
def load_qa(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# ═══ Enhanced Judge System Prompts (Category-specific) ═══
JUDGE_SYS_BASE = (
    "You evaluate answer quality. Score 0.0 to 1.0. "
    "Criteria: Correctness (factually matches expected), "
    "Completeness (covers all key points), "
    "Precision (no wrong information). "
    "Reply with ONLY a single decimal number between 0.0 and 1.0, nothing else.")

JUDGE_SYS_MAP = {
    "A": JUDGE_SYS_BASE + "\nFor selection questions, focus on whether the specific fact is correctly identified.",
    "B": JUDGE_SYS_BASE + "\nFor enumeration/counting questions, score based on COVERAGE: how many of the expected items are present in the answer. If the count is wrong, score 0.3 or lower. Missing items should heavily penalize the score.",
    "C": JUDGE_SYS_BASE + "\nFor aggregation/summary questions, evaluate whether ALL key themes and subtopics are covered. Partial coverage should be scored proportionally.",
    "D": JUDGE_SYS_BASE + "\nFor multi-hop questions, verify that facts from MULTIPLE sections are connected correctly. Missing cross-references should reduce the score significantly.",
}

# ═══ Token-level F1 Computation ═══
def compute_token_f1(expected: str, actual: str) -> float:
    """Compute token-level F1 score between expected and actual answers."""
    import re
    def tokenize(text):
        return set(re.findall(r"\w+", text.lower()))
    
    expected_tokens = tokenize(expected)
    actual_tokens = tokenize(actual)
    
    if not expected_tokens or not actual_tokens:
        return 0.0
    
    common = expected_tokens & actual_tokens
    if not common:
        return 0.0
    
    precision = len(common) / len(actual_tokens)
    recall = len(common) / len(expected_tokens)
    
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0


# ═══ Evidence Accuracy (key fact overlap) ═══
def compute_evidence_accuracy(expected: str, actual: str) -> float:
    """Check how many key facts from expected answer appear in actual answer.
    Uses sentence-level matching.
    """
    import re
    def extract_facts(text):
        # Split by sentence boundaries and filter short fragments
        sentences = re.split(r"[.!?;]\s+|\n", text)
        return [s.strip().lower() for s in sentences if len(s.strip()) > 10]
    
    expected_facts = extract_facts(expected)
    if not expected_facts:
        return 1.0  # No facts to check
    
    actual_lower = actual.lower()
    matched = 0
    for fact in expected_facts:
        # Extract key n-grams from the fact
        words = re.findall(r"\w+", fact)
        if len(words) < 3:
            continue
        # Check if significant portion of words appear in actual
        word_hits = sum(1 for w in words if w in actual_lower)
        if word_hits / len(words) >= 0.5:  # 50% threshold
            matched += 1
    
    total = len([f for f in expected_facts if len(re.findall(r"\w+", f)) >= 3])
    return matched / total if total > 0 else 1.0


# ═══ MULTI-PROVIDER JUDGE ═══
class JudgeLLM:
    """Separate LLM client for judging — uses Gemini for better objectivity."""

    def __init__(self):
        self.provider = JUDGE_PROVIDER
        self.calls = 0
        self._client = None

        if self.provider == "gemini":
            import google.generativeai as genai
            genai.configure(api_key=os.environ["GEMINI_API_KEY"])
            self._client = genai.GenerativeModel("gemini-2.0-flash")
            print("Judge: Gemini 2.0 Flash (free, fast)")

        elif self.provider == "groq":
            from openai import OpenAI
            self._client = OpenAI(
                api_key=os.environ["GROQ_API_KEY"],
                base_url="https://api.groq.com/openai/v1")
            print("Judge: Groq Llama 3.3 70B (free, ultra-fast)")

        else:
            print("Judge: Mistral Small (same as agent)")

    def score(self, prompt, system_prompt=None):
        self.calls += 1
        sys_prompt = system_prompt or JUDGE_SYS_BASE
        try:
            if self.provider == "gemini":
                resp = self._client.generate_content(prompt)
                return resp.text.strip()

            elif self.provider == "groq":
                resp = self._client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[{"role": "system", "content": sys_prompt},
                              {"role": "user", "content": prompt}],
                    max_tokens=10, temperature=0.0)
                return resp.choices[0].message.content.strip()

            else:
                return llm.chat(prompt, system=sys_prompt,
                                max_tokens=10, temperature=0.0).strip()
        except Exception as e:
            print(f"  Judge error: {e}")
            return "0.5"


judge_llm = JudgeLLM()


def judge(question, expected, actual, category="A"):
    """Enhanced judge with category-specific prompts."""
    NL = chr(10)
    sys_prompt = JUDGE_SYS_MAP.get(category, JUDGE_SYS_BASE)
    prompt = (f"{sys_prompt}{NL}{NL}"
              f"Question: {question}{NL}{NL}"
              f"Expected answer: {expected}{NL}{NL}"
              f"Actual answer: {actual[:2000]}{NL}{NL}"
              f"Score (0.0-1.0):")
    txt = judge_llm.score(prompt, system_prompt=sys_prompt)
    try:
        import re
        match = re.search(r"(\d\.\d+|\d)", txt)
        if match:
            return max(0.0, min(1.0, float(match.group())))
        return max(0.0, min(1.0, float(txt)))
    except Exception:
        return 0.5

print("Enhanced Judge defined (provider: " + JUDGE_PROVIDER + ", category-specific prompts)")
print("Additional metrics: Token-level F1, Evidence Accuracy")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_eval_for_doc(doc_key, qa_path, tree, vs, llm, agent_override=None, max_workers=3):
    questions = load_qa(qa_path)["questions"]
    print(f"\nEvaluating {doc_key}: {len(questions)} questions")
    
    def eval_one(item):
        qid, cat, question, expected = item["id"], item["category"], item["question"], item["answer"]
        
        if agent_override:
            agent = agent_override
        else:
            agent = Agent(tree, vs, llm)
        
        t0 = time.time()
        calls_before = llm.calls
        try:
            actual = agent.query(question, verbose=False)
        except Exception as e:
            actual = f"Error: {e}"
        latency = time.time() - t0
        
        # LLM-as-Judge score (category-specific)
        llm_judge_score = judge(question, expected, actual, category=cat)
        
        # Token-level F1
        token_f1 = compute_token_f1(expected, actual)
        
        # Evidence Accuracy
        evidence_acc = compute_evidence_accuracy(expected, actual)
        
        calls_used = llm.calls - calls_before
        
        tag = "PASS" if llm_judge_score >= 0.7 else "WARN" if llm_judge_score >= 0.4 else "FAIL"
        print(f"  {tag} [{qid}] judge={llm_judge_score:.2f} f1={token_f1:.2f} acc={evidence_acc:.2f}  {latency:.1f}s")
        
        return {
            "doc": doc_key, "id": qid, "category": cat,
            "question": question, "expected": expected,
            "actual": actual,
            "llm_judge_score": llm_judge_score,
            "token_f1": token_f1,
            "evidence_accuracy": evidence_acc,
            "latency": latency, "llm_calls": calls_used
        }
    
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(eval_one, q): q for q in questions}
        for f in as_completed(futures):
            results.append(f.result())
    
    return sorted(results, key=lambda r: r["id"])
    

print("Enhanced evaluation runner defined (LLM Judge + Token F1 + Evidence Accuracy)")

### 7.1 Execute Evaluation (Single vs Multi-Agent)

Set `AGENT_TYPE` to compare:
- `"single"` — baseline single ReAct agent
- `"multi"` — Planner→Workers→Synthesizer
- `"adaptive"` — A→Single, BCD→Multi (recommended)

In [ ]:
AGENT_MAX_ITER = 25  # v5: room for dual-phase + verification

In [ ]:
AGENT_TYPE = "single"  # "single" | "multi" | "adaptive"
llm = LLM()
all_results = []

In [ ]:
# 에이전트 설정
key = "ai_act"
tree, vs = indices[key]["tree"], indices[key]["vs"]
if AGENT_TYPE == "single": agent_obj = Agent(tree, vs, llm)
elif AGENT_TYPE == "multi": agent_obj = MultiAgent(tree, vs, llm)
else: agent_obj = AdaptiveAgent(tree, vs, llm)

# 테스트 질문
test_questions = [
    "How many articles are there in the AI Act?",
    "How many titles are there in the AI Act?",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    answer = agent_obj.query(q, verbose=True)
    print(f"A: {answer}")

In [ ]:
import re

# 1) Title이 실제로 어떻게 쓰여있는지 확인
print("=== 'Title' 포함 라인 샘플 ===")
count = 0
for node in tree.flatten():
    text = node.all_text() if hasattr(node, 'all_text') else ""
    for line in text.split('\n'):
        line = line.strip()
        if line.lower().startswith('title') and len(line) < 150:
            print(f"  {line}")
            count += 1
            if count >= 20:
                break
    if count >= 20:
        break

# 2) Article 고유 개수 확인 (중복 제거)
all_articles = set()
for node in tree.flatten():
    text = node.all_text() if hasattr(node, 'all_text') else ""
    found = re.findall(r'(?m)^(Article\s+\d+)', text)
    all_articles.update(found)

print(f"\n=== Article 고유 개수: {len(all_articles)} ===")
sorted_articles = sorted(all_articles, key=lambda x: int(re.search(r'\d+', x).group()))
for a in sorted_articles:
    print(f"  {a}")

In [ ]:
import re

# 전체 텍스트에서 "Article XX" 패턴 추출
tb = ToolBox(tree, vs)
all_articles = set()
for nid, node in tb._map.items():
    text = node.all_text() if hasattr(node, 'all_text') else ""
    # "Article 1", "Article 85" 같은 패턴 찾기
    found = re.findall(r'\bArticle\s+(\d+)\b', text)
    all_articles.update(int(a) for a in found)

print(f"본문에서 발견된 Article 번호: {min(all_articles)} ~ {max(all_articles)}")
print(f"고유 Article 수: {len(all_articles)}")
print(f"목록: {sorted(all_articles)}")

In [ ]:
with open(DATA_DIR / "eval_results_improved.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"Evaluation complete: {len(all_results)} questions ({AGENT_TYPE} agent)")
print(f"Total LLM calls: {llm.calls}")
print(f"Tokens: ~{llm.total_in:,} in / ~{llm.total_out:,} out")

## 8. Results Analysis & Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.size"] = 11
import numpy as np

with open(DATA_DIR / "eval_results_improved.json", "r") as f:
    all_results = json.load(f)

cat_names = {"A": "Selection", "B": "Enumeration", "C": "Aggregation", "D": "Multi-hop"}
doc_names = {"gen_ai": "Gen AI Use Cases", "ai_act": "EU AI Act", "parks": "National Parks"}

print(f"Loaded {len(all_results)} results")

### 8.1 Overall Summary Table

In [ ]:
print(all_results[0].keys())

In [ ]:
# ═══ Summary with all 3 metrics ═══
summary_data = {}
docs_list, cats_list = [], []

for r in all_results:
    dk, cat = r["doc"], r["category"]
    key = (dk, cat)
    if dk not in docs_list: docs_list.append(dk)
    if cat not in cats_list: cats_list.append(cat)
    if key not in summary_data:
        summary_data[key] = {"scores": [], "f1s": [], "accs": [], "count": 0}
    summary_data[key]["scores"].append(r.get("llm_judge_score", r.get("score", 0)))
    summary_data[key]["f1s"].append(r.get("token_f1", 0))
    summary_data[key]["accs"].append(r.get("evidence_accuracy", 0))
    summary_data[key]["count"] += 1

for k in summary_data:
    d = summary_data[k]
    d["avg_score"] = sum(d["scores"]) / len(d["scores"])
    d["avg_f1"] = sum(d["f1s"]) / len(d["f1s"])
    d["avg_acc"] = sum(d["accs"]) / len(d["accs"])

cats_list = sorted(cats_list)

# Print enhanced summary table
print("=" * 100)
print(f"  {'Document':<15} {'Category':<15} {'LLM Judge':>12} {'Token F1':>12} {'Evidence Acc':>12} {'Count':>8}")
print("=" * 100)
for dk in docs_list:
    for cat in cats_list:
        d = summary_data.get((dk, cat), {})
        if d:
            print(f"  {doc_names.get(dk, dk):<15} {cat} ({cat_names.get(cat, cat):<12}) "
                  f"{d['avg_score']:>10.3f} {d['avg_f1']:>12.3f} {d['avg_acc']:>12.3f} {d['count']:>8}")

# Overall averages
all_scores = [r.get("llm_judge_score", r.get("score", 0)) for r in all_results]
all_f1s = [r.get("token_f1", 0) for r in all_results]
all_accs = [r.get("evidence_accuracy", 0) for r in all_results]
print("=" * 100)
print(f"  {'OVERALL':<30} {sum(all_scores)/len(all_scores):>12.3f} "
      f"{sum(all_f1s)/len(all_f1s):>12.3f} {sum(all_accs)/len(all_accs):>12.3f} {len(all_results):>8}")
print("=" * 100)

### 8.2 Score Heatmap: Document x Category

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

docs_list = list(DOCS.keys())
cats_list = ["A", "B", "C", "D"]
matrix = np.zeros((len(docs_list), len(cats_list)))

for i, dk in enumerate(docs_list):
    for j, cat in enumerate(cats_list):
        key = (dk, cat)
        if key in summary_data:
            matrix[i, j] = summary_data[key]["avg_score"]

im = ax.imshow(matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(cats_list)))
ax.set_xticklabels([f"{c} {cat_names[c]}" for c in cats_list])
ax.set_yticks(range(len(docs_list)))
ax.set_yticklabels([doc_names.get(d, d) for d in docs_list])

for i in range(len(docs_list)):
    for j in range(len(cats_list)):
        color = "white" if matrix[i, j] < 0.5 else "black"
        ax.text(j, i, f"{matrix[i,j]:.2f}", ha="center", va="center",
                fontsize=14, fontweight="bold", color=color)

plt.colorbar(im, ax=ax, label="Average Score")
ax.set_title("Score Heatmap: Document x Question Category", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(str(DATA_DIR / "heatmap.png"), dpi=150)
plt.show()

### 8.3 Average Score by Category and Document

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: by category
cat_scores = {}
for cat in "ABCD":
    s = [r["score"] for r in all_results if r["category"] == cat]
    cat_scores[cat] = sum(s) / len(s) if s else 0

colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]
labels_cat = [f"{c} {cat_names[c]}" for c in "ABCD"]
vals_cat = [cat_scores[c] for c in "ABCD"]

bars = axes[0].bar(labels_cat, vals_cat, color=colors, edgecolor="white", linewidth=1.5)
for bar, c in zip(bars, "ABCD"):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{cat_scores[c]:.2f}", ha="center", fontweight="bold", fontsize=12)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel("Average Score")
axes[0].set_title("Score by Question Category")
axes[0].axhline(y=0.7, color="gray", linestyle="--", alpha=0.5, label="Good (0.7)")
axes[0].legend()

# Right: by document
doc_scores = {}
for dk in DOCS:
    s = [r["score"] for r in all_results if r["doc"] == dk]
    doc_scores[dk] = sum(s) / len(s) if s else 0

doc_labels = [doc_names.get(d, d) for d in DOCS]
doc_vals = [doc_scores[d] for d in DOCS]
doc_colors = ["#9b59b6", "#1abc9c", "#f39c12"]

bars2 = axes[1].bar(doc_labels, doc_vals, color=doc_colors, edgecolor="white", linewidth=1.5)
for bar, dk in zip(bars2, DOCS):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{doc_scores[dk]:.2f}", ha="center", fontweight="bold", fontsize=12)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel("Average Score")
axes[1].set_title("Score by Document")
axes[1].axhline(y=0.7, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(str(DATA_DIR / "scores_overview.png"), dpi=150)
plt.show()

### 8.4 Latency & Token Usage Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]
cat_labels = [f"{c} {cat_names[c]}" for c in "ABCD"]

# Left: latency boxplot
cat_lats = {c: [r["latency"] for r in all_results if r["category"] == c] for c in "ABCD"}
bp = axes[0].boxplot([cat_lats[c] for c in "ABCD"], labels=cat_labels, patch_artist=True)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
axes[0].set_ylabel("Latency (seconds)")
axes[0].set_title("Response Latency by Category")

# Right: token boxplot
cat_toks = {c: [r["in_tokens"] + r["out_tokens"] for r in all_results if r["category"] == c] for c in "ABCD"}
bp2 = axes[1].boxplot([cat_toks[c] for c in "ABCD"], labels=cat_labels, patch_artist=True)
for patch, c in zip(bp2["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
axes[1].set_ylabel("Total Tokens (in + out)")
axes[1].set_title("Token Usage by Category")

plt.tight_layout()
plt.savefig(str(DATA_DIR / "perf_analysis.png"), dpi=150)
plt.show()

### 8.5 Detailed: Lowest-Scoring Questions

In [ ]:
print("=" * 80)
print("LOWEST-SCORING QUESTIONS (score < 0.5)")
print("=" * 80)

low = sorted([r for r in all_results if r["score"] < 0.5], key=lambda x: x["score"])
for r in low[:15]:
    print(f"  [{r['id']}] {r['doc']} | Cat {r['category']} | Score: {r['score']:.2f}")
    print(f"    Q: {r['question'][:90]}")
    print(f"    Expected: {r['expected'][:90]}")
    print(f"    Got:      {r['actual'][:90]}")
    print()

print(f"Total low-scoring: {len(low)} / {len(all_results)}")

### 8.6 Grouped Comparison: Document x Category

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(cats_list))
width = 0.25
doc_clrs = {"gen_ai": "#9b59b6", "ai_act": "#1abc9c", "parks": "#f39c12"}

for i, dk in enumerate(docs_list):
    vals = [summary_data.get((dk, c), {}).get("avg_score", 0) for c in cats_list]
    bars = ax.bar(x + i * width, vals, width,
                  label=doc_names.get(dk, dk), color=doc_clrs.get(dk, "gray"), edgecolor="white")
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{v:.2f}", ha="center", fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels([f"{c} - {cat_names[c]}" for c in cats_list])
ax.set_ylim(0, 1.15)
ax.set_ylabel("Average Score")
ax.set_title("Document x Category Score Comparison", fontsize=13)
ax.legend(loc="upper right")
ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(str(DATA_DIR / "grouped_comparison.png"), dpi=150)
plt.show()

### 8.7 Radar Chart per Document

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw=dict(projection="polar"))

angles = np.linspace(0, 2 * np.pi, 4, endpoint=False).tolist()
angles += angles[:1]

doc_clr_list = ["#9b59b6", "#1abc9c", "#f39c12"]

for idx, dk in enumerate(docs_list):
    vals = [summary_data.get((dk, c), {}).get("avg_score", 0) for c in "ABCD"]
    vals += vals[:1]
    ax = axes[idx]
    ax.fill(angles, vals, alpha=0.25, color=doc_clr_list[idx])
    ax.plot(angles, vals, "o-", color=doc_clr_list[idx], linewidth=2)
    ax.set_xticks(angles[:-1])
    cat_radar_labels = [f"{c} {cat_names[c]}" for c in "ABCD"]
    ax.set_xticklabels(cat_radar_labels, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title(doc_names.get(dk, dk), fontsize=11, pad=15)

plt.suptitle("Score Profile per Document", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(str(DATA_DIR / "radar.png"), dpi=150, bbox_inches="tight")
plt.show()

### 8.8 Single vs Multi-Agent Comparison

To compare, run the evaluation twice:
1. Set `AGENT_TYPE = "single"` → run → save results as `eval_single.json`
2. Set `AGENT_TYPE = "adaptive"` → run → save results as `eval_adaptive.json`

Then run the comparison below.

In [ ]:
# ═══ COMPARISON: Single vs Multi-Agent ═══
# After running both, load and compare

import os

single_path = DATA_DIR / "eval_single.json"
adaptive_path = DATA_DIR / "eval_adaptive.json"

# Save current results under the agent type name
current_results_path = DATA_DIR / f"eval_{AGENT_TYPE}.json"
shutil.copy(DATA_DIR / "eval_results.json", current_results_path)
print(f"Saved current results as eval_{AGENT_TYPE}.json")

# If both exist, generate comparison
if single_path.exists() and adaptive_path.exists():
    with open(single_path) as f:
        single_res = json.load(f)
    with open(adaptive_path) as f:
        adaptive_res = json.load(f)

    print(f"{'Category':<15} {'Single':>10} {'Adaptive':>10} {'Delta':>10}")
    print("-" * 50)

    for cat in "ABCD":
        s_scores = [r["score"] for r in single_res if r["category"] == cat]
        a_scores = [r["score"] for r in adaptive_res if r["category"] == cat]
        s_avg = sum(s_scores) / len(s_scores) if s_scores else 0
        a_avg = sum(a_scores) / len(a_scores) if a_scores else 0
        delta = a_avg - s_avg
        arrow = "+" if delta > 0 else ""
        print(f"  {cat} {cat_names[cat]:<12} {s_avg:>8.3f}   {a_avg:>8.3f}   {arrow}{delta:>8.3f}")

    s_all = sum(r["score"] for r in single_res) / len(single_res)
    a_all = sum(r["score"] for r in adaptive_res) / len(adaptive_res)
    print(f"{'OVERALL':<15} {s_all:>10.3f} {a_all:>10.3f} {'+' if a_all > s_all else ''}{a_all - s_all:>8.3f}")

    # Bar comparison chart
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(4)
    w = 0.35
    s_vals = []
    a_vals = []
    for cat in "ABCD":
        s = [r["score"] for r in single_res if r["category"] == cat]
        a = [r["score"] for r in adaptive_res if r["category"] == cat]
        s_vals.append(sum(s)/len(s) if s else 0)
        a_vals.append(sum(a)/len(a) if a else 0)

    bars1 = ax.bar(x - w/2, s_vals, w, label="Single Agent", color="#e74c3c", alpha=0.8)
    bars2 = ax.bar(x + w/2, a_vals, w, label="Adaptive (Multi)", color="#2ecc71", alpha=0.8)

    for bar, v in zip(bars1, s_vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f"{v:.2f}",
                ha="center", fontsize=10)
    for bar, v in zip(bars2, a_vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f"{v:.2f}",
                ha="center", fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels([f"{c} {cat_names[c]}" for c in "ABCD"])
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Average Score")
    ax.set_title("Single Agent vs Adaptive Multi-Agent", fontsize=13)
    ax.legend()
    ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(str(DATA_DIR / "single_vs_multi.png"), dpi=150)
    plt.show()
else:
    print(f"Run evaluation with both agent types to compare.")
    print(f"  Exists: single={single_path.exists()}, adaptive={adaptive_path.exists()}")

## 9. Final Report Summary

In [ ]:
print("=" * 70)
print("  AICOSS 26 CAPSTONE - EVALUATION REPORT")
print("=" * 70)

all_s = [r["score"] for r in all_results]
all_l = [r["latency"] for r in all_results]

print(f"  Overall Results ({len(all_results)} questions)")
print(f"    Average Score:   {sum(all_s)/len(all_s):.3f}")
print(f"    Average Latency: {sum(all_l)/len(all_l):.1f}s")

print(f"  By Category:")
for cat in "ABCD":
    s = [r["score"] for r in all_results if r["category"] == cat]
    l = [r["latency"] for r in all_results if r["category"] == cat]
    cname = cat_names[cat]
    print(f"    {cat} ({cname:15s}):  score={sum(s)/len(s):.3f}  latency={sum(l)/len(l):.1f}s  n={len(s)}")

print(f"  By Document:")
for dk in DOCS:
    s = [r["score"] for r in all_results if r["doc"] == dk]
    dname = doc_names.get(dk, dk)
    print(f"    {dname:22s}:  score={sum(s)/len(s):.3f}  n={len(s)}")

good = sum(1 for r in all_results if r["score"] >= 0.7)
mid  = sum(1 for r in all_results if 0.4 <= r["score"] < 0.7)
bad  = sum(1 for r in all_results if r["score"] < 0.4)
total = len(all_results)
print(f"  Score Distribution:")
print(f"    Good  (>=0.7): {good:3d} ({good/total*100:.0f}%)")
print(f"    Medium (0.4-0.7): {mid:3d} ({mid/total*100:.0f}%)")
print(f"    Low   (<0.4):  {bad:3d} ({bad/total*100:.0f}%)")

print(f"  Total API Usage:")
print(f"    Agent LLM calls: {llm.calls}")
print(f"    Judge LLM calls: {judge_llm.calls} ({judge_llm.provider})")
print(f"    Tokens: ~{llm.total_in:,} in + ~{llm.total_out:,} out")
print("=" * 70)